# PPO Implementation

In [1]:
import os
import sys
import time
import random
import numpy as np
import matplotlib.pyplot as plt

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print('Using project root:', project_root)

Using project root: /Users/rithvik/Documents/hnrs/Decoder


In [9]:
from ldpc.bp_decoder import BpDecoder
from utils.LDPC_encode import QCLDPCEncoder
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER
from ppo.ppo_env import PpoEnv
from ppo.ppo_agent import PpoAgent
from ppo.ppo_decoder import PpoDecoder

In [3]:
H = np.loadtxt(os.path.join(project_root, 'pc_matrices', 'WRAN_irreg_384_256.csv'), delimiter=',', dtype=int)

encoder = QCLDPCEncoder(H=H)
m, n = H.shape
k = n - m

clusters = np.arange(m).reshape(8, -1)

print(f'H shape: {H.shape}, K={encoder.K}, rate={encoder.K / encoder.N:.3f}')
print('Clusters:')
print(clusters)

Initializing Encoder from H: Full Matrix Size 128x384, Message Bits: 256
  > Inverting Parity Matrix (this may take a moment for large Z)...
  > Computing Generator Matrix...
Encoder Ready.
H shape: (128, 384), K=256, rate=0.667
Clusters:
[[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15]
 [ 16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31]
 [ 32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47]
 [ 48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63]
 [ 64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79]
 [ 80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95]
 [ 96  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111]
 [112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127]]


---
---

In [4]:
def generate_data(encoder, n_frames, snr_db, seed=42):
    rng = np.random.RandomState(seed)
    K, N = encoder.K, encoder.N
    messages = rng.randint(0, 2, size=(n_frames, K))
    codewords = encoder.encode(messages)
    bpsk = 1.0 - 2.0 * codewords.astype(np.float64)
    llr_matrix = AWGNChannel(bpsk, snr_db)
    return codewords, llr_matrix


TRAIN_SNR_DB = 4.0
N_TRAIN = 10000

train_codewords, train_llrs = generate_data(
    encoder, N_TRAIN, TRAIN_SNR_DB, seed=SEED
)

### Sanity Check

I initialized two agents. One agent i will train for 10,000 frames and another will not be trained at all. I will then compare the decoding performance of both the agents

In [5]:
bp_dec = BpDecoder(H, schedule="cluster")
env = PpoEnv(H, clusters, bp_dec, l_max=m)

agent = PpoAgent(
    obs_dim=n,
    num_clusters=len(clusters),
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_eps=0.2,
    ppo_epochs=4,
    minibatch_size=64,
    entropy_coeff=0.01,
    value_coeff=0.5,
)

In [12]:
agent_dumb = PpoAgent(
    obs_dim=n,
    num_clusters=len(clusters),
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_eps=0.2,
    ppo_epochs=4,
    minibatch_size=64,
    entropy_coeff=0.01,
    value_coeff=0.5,
)

### Training of agent

In [6]:
t0 = time.time()
rewards = agent.train(
    env,
    llr_list=[train_llrs[i] for i in range(N_TRAIN)],
    codeword_list=[train_codewords[i] for i in range(N_TRAIN)],
    update_every=10,
    verbose=True,
)
t_train = time.time() - t0
print(f"  Training time: {t_train:.1f}s")
print(f"  Final avg reward (last 50): {np.mean(rewards[-50:]):.4f}")

[PpoAgent] Training on device: cpu
  Episode 10/10000 | Avg reward: 42.0059 | π loss: -0.0126 | V loss: 103.8342 | entropy: 2.0761
  Episode 20/10000 | Avg reward: 60.8765 | π loss: -0.0166 | V loss: 75.6191 | entropy: 2.0753
  Episode 30/10000 | Avg reward: 31.1299 | π loss: -0.0165 | V loss: 37.7901 | entropy: 2.0753
  Episode 40/10000 | Avg reward: 41.3172 | π loss: -0.0192 | V loss: 48.1856 | entropy: 2.0735
  Episode 50/10000 | Avg reward: 42.8083 | π loss: -0.0128 | V loss: 51.8710 | entropy: 2.0740
  Episode 60/10000 | Avg reward: 40.9787 | π loss: -0.0183 | V loss: 46.7157 | entropy: 2.0696
  Episode 70/10000 | Avg reward: 31.0373 | π loss: -0.0232 | V loss: 42.8766 | entropy: 2.0705
  Episode 80/10000 | Avg reward: 49.1370 | π loss: -0.0228 | V loss: 57.2946 | entropy: 2.0678
  Episode 90/10000 | Avg reward: 44.8988 | π loss: -0.0301 | V loss: 49.5894 | entropy: 2.0550
  Episode 100/10000 | Avg reward: 36.1599 | π loss: -0.0256 | V loss: 33.7853 | entropy: 2.0673
  Episode 110

### Generating test data

In [8]:
TEST_FRAMES = 10000

messages_test = np.random.randint(0, 2, size=(TEST_FRAMES, k))
codewords_test = encoder.encode(messages_test)
tx_cw = 1.0 - 2.0 * codewords_test

### Performance of trained agent

In [10]:
TEST_SNRS = [3.0, 4.0, 5.0]
decoder = BpDecoder(H, schedule="cluster")
I_MAX = 10

ppo_bers = []

for snr_db in TEST_SNRS:
    rx_llrs = AWGNChannel(tx_cw, snr_db)
    decoded_cw = []

    for i in range(TEST_FRAMES):
        llr = rx_llrs[i, :]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        for it in range(I_MAX*len(clusters)):
            action, _, _ = agent.select_action(llr, training=False)
            llr = decoder.decode_cluster(clusters[action])
        
        decoded_cw.append((llr < 0).astype(int))
    
    decoded_cw = np.array(decoded_cw)
    decoded_msgs = decoded_cw[:, :k]
    ppo_ber = findBER(messages_test, decoded_msgs)
    ppo_bers.append(ppo_ber)
    print(f"SNR={snr_db}dB: PPO BER = {ppo_ber:.4e}")

SNR=3.0dB: PPO BER = 7.6019e-02
SNR=4.0dB: PPO BER = 5.2414e-02
SNR=5.0dB: PPO BER = 3.2352e-02


### Performance of untrained agent

In [13]:
ppo_dumb_bers = []
decoder_dumb = BpDecoder(H, schedule="cluster")

for snr_db in TEST_SNRS:
    rx_llrs = AWGNChannel(tx_cw, snr_db)
    decoded_cw = []

    for i in range(TEST_FRAMES):
        llr = rx_llrs[i, :]

        decoder_dumb.reset()
        decoder_dumb.initialise_log_domain_bp(llr)
        for it in range(I_MAX*len(clusters)):
            action, _, _ = agent_dumb.select_action(llr, training=False)
            llr = decoder_dumb.decode_cluster(clusters[action])
        
        decoded_cw.append((llr < 0).astype(int))
    
    decoded_cw = np.array(decoded_cw)
    decoded_msgs = decoded_cw[:, :k]
    ppo_ber = findBER(messages_test, decoded_msgs)
    ppo_dumb_bers.append(ppo_ber)
    print(f"SNR={snr_db}dB: PPO BER = {ppo_ber:.4e}")

SNR=3.0dB: PPO BER = 7.5734e-02
SNR=4.0dB: PPO BER = 5.1884e-02
SNR=5.0dB: PPO BER = 3.1458e-02


As we can see the performance, the performance of both the trained and untrained agent is the same, which means there is something wrong in implementation

### Random Scheduler

As another test I tested with random scheduling of clusters

In [11]:
TEST_SNRS = [3.0, 4.0, 5.0]
decoder_rand = BpDecoder(H, schedule="cluster")
I_MAX = 10

rand_bers = []

for snr_db in TEST_SNRS:
    rx_llrs = AWGNChannel(tx_cw, snr_db)
    decoded_cw = []

    for i in range(TEST_FRAMES):
        llr = rx_llrs[i, :]

        decoder_rand.reset()
        decoder_rand.initialise_log_domain_bp(llr)
        for it in range(I_MAX*len(clusters)):
            action = np.random.randint(0, len(clusters))
            llr = decoder_rand.decode_cluster(clusters[action])
        
        decoded_cw.append((llr < 0).astype(int))
    
    decoded_cw = np.array(decoded_cw)
    decoded_msgs = decoded_cw[:, :k]
    rand_ber = findBER(messages_test, decoded_msgs)
    rand_bers.append(rand_ber)
    print(f"SNR={snr_db}dB: Random BER = {rand_ber:.4e}")

SNR=3.0dB: Random BER = 3.6433e-02
SNR=4.0dB: Random BER = 4.2352e-03
SNR=5.0dB: Random BER = 1.4414e-04


The performance of random agent is way better than the performance of ppo agent